In [ ]:
import requests
import time
import pandas as pd
import json
from pathlib import Path
from typing import Optional, Dict, List
from ../src/id_converter import url_id_to_numeric, numeric_to_url_id

In [ ]:
def fetch_event(event_id: int, timeout: int = 10) -> Optional[dict]:
    str_event_id = numeric_to_url_id(event_id)
    url = f"https://results.advancedeventsystems.com/api/event/{str_event_id}"
    
    try:
        response = requests.get(url, timeout=timeout)
        
        if response.status_code != 200:
            return None
        
        data = response.json()
        
        if not data or 'EventId' not in data:
            return None
        
        return data
    
    except requests.RequestException:
        return None

In [ ]:
def save_event_json(data: dict, base_path: str = "Data/raw/events") -> None:
    """
    Save raw event JSON to disk using event_id as filename.
    """
    event_id = data.get("EventId")
    
    if not event_id:
        return
    
    # Ensure directory exists
    Path(base_path).mkdir(parents=True, exist_ok=True)
    
    file_path = Path(base_path) / f"event_{event_id}.json"
    
    with open(file_path, "w") as f:
        json.dump(data, f, indent=2)

In [ ]:
def fetch_and_save_event(event_id: int) -> Optional[dict]:
    data = fetch_event(event_id)
    
    if data:
        save_event_json(data)
    
    return data

In [ ]:
for event_id in range(40982, 44444):
    print(f"Processing {event_id}...")
    
    data = fetch_and_save_event(event_id)
    
    if data is None:
        print(f"Event {event_id} not found")